In [ ]:
# Si no tiene scikit-image instalada ejecutar la celda siguiente
# pip install scikit-image

In [1]:
# Importaciones
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os
import math
import glob
import shutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, Input
from tensorflow.keras.utils import image_dataset_from_directory
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

In [2]:
# ============================================================
# CONFIGURACIÓN DE GPU PARA TENSORFLOW/KERAS
# ============================================================

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs físicas detectadas:", gpus)

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

        logical_gpus = tf.config.list_logical_devices("GPU")
        print(f"{len(gpus)} GPU física(s), {len(logical_gpus)} GPU lógica(s) configurada(s).")

    except RuntimeError as e:
        print("Error configurando la GPU:", e)
else:
    print("TensorFlow NO detectó GPU. El notebook correrá en CPU.")

TensorFlow version: 2.21.0
GPUs físicas detectadas: []
TensorFlow NO detectó GPU. El notebook correrá en CPU.


# **Fase 1: Implementaciones de Matriz Embedding y Extracciones de Mensajes**

## **1.1. Código Hamming (7,4)**

### **1.1.1. Implementación de Matrix Embedding con Código Hamming (7,4)**

El objetivo del Matrix Embedding es minimizar la cantidad de modificaciones necesarias en la imagen de cobertura, reduciendo así la distorsión estadística. Utilizando un código Hamming (7,4), podemos ocultar 3 bits de información secreta en un bloque de 7 píxeles alterando, como máximo, un solo bit menos significativo (LSB).

Matemáticamente, definimos la matriz de comprobación de paridad $H$ de dimensiones $3 \times 7$. Para un bloque de píxeles $x$, calculamos su síndrome actual. Si la diferencia entre el mensaje secreto y el síndrome actual no es nula, el vector resultante nos indica exactamente el índice de la columna en $H$ (y por ende, el píxel) que debe ser modificado para que el nuevo síndrome coincida con el secreto.

In [4]:
def embed_hamming_7_3(pixels_block, secret_bits):
    """
    Oculta 3 bits secretos en un bloque de 7 píxeles.
    pixels_block: array de numpy con 7 valores (ej. [150, 151, 149...])
    secret_bits: array de numpy con 3 bits (ej. [1, 0, 1])
    """
    # 1. Matriz de paridad H para Hamming (7,4)
    H = np.array([
        [0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1]
    ])

    # 2. Extraer los LSB (Bits Menos Significativos) del bloque de píxeles
    x = pixels_block % 2

    # 3. Calcular el síndrome actual: s_current = H * x^T (mod 2)
    s_current = np.dot(H, x) % 2

    # 4. Calcular la diferencia con el secreto que queremos: d = secret - s_current (mod 2)
    d = (secret_bits - s_current) % 2

    # 5. Encontrar qué píxel debemos cambiar
    # Convertimos el vector 'd' a un número entero (que coincide con la columna en H)
    index_to_change = d[0]*4 + d[1]*2 + d[2]*1

    modified_pixels = pixels_block.copy()

    # Si el índice es 0, la imagen ya tiene el mensaje por pura casualidad (¡No hacemos nada!)
    # Si es > 0, alteramos el LSB del píxel en la posición correspondiente
    if index_to_change > 0:
        idx = index_to_change - 1

        # Invertir el LSB: si es par suma 1, si es impar resta 1
        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels

## **1.1.2. Extracción del Mensaje con Código Hamming (7,4)**

La principal ventaja de este esquema es la eficiencia en la recuperación. Para extraer el mensaje de la imagen stego, el receptor no necesita conocer la imagen original. Basta con extraer los LSB de los bloques de 7 píxeles y multiplicarlos por la misma matriz de comprobación de paridad $H$. El síndrome resultante será exactamente el mensaje secreto incrustado.

In [5]:
def extract_hamming_7_3(stego_pixels_block):
    """
    Extrae los 3 bits secretos ocultos en un bloque de 7 píxeles stego.
    stego_pixels_block: array de numpy con 7 valores modificados.
    """
    H = np.array([
        [0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1]
    ])

    # 1. Extraer los LSB del bloque stego
    x_stego = stego_pixels_block % 2

    # 2. El mensaje secreto es simplemente el síndrome del bloque
    extracted_bits = np.dot(H, x_stego) % 2

    return extracted_bits

### **1.1.3. Validación Funcional: Inserción y Extracción con Hamming (7,4)**

Para comprobar la correctitud matemática de las operaciones algebraicas de los síndromes, se diseñó una prueba unitaria aislada. Se inyectó el vector secreto [1, 0, 1] en un bloque de control de 7 píxeles.

In [6]:
# --- Bloque de Prueba Unitaria ---
print("--- Prueba de Funcionamiento ---")
original_pixels = np.array([150, 151, 149, 152, 153, 154, 155])
secret = np.array([1, 0, 1])

print(f"Píxeles originales: {original_pixels}")
print(f"Secreto a ocultar:  {secret}")

# Proceso de embedding
stego_pixels = embed_hamming_7_3(original_pixels, secret)
print(f"Píxeles stego:      {stego_pixels}")

# Proceso de extracción
recovered_secret = extract_hamming_7_3(stego_pixels)
print(f"Secreto recuperado: {recovered_secret}")
print(f"¿Éxito?: {'Sí' if np.array_equal(secret, recovered_secret) else 'No'}")

--- Prueba de Funcionamiento ---
Píxeles originales: [150 151 149 152 153 154 155]
Secreto a ocultar:  [1 0 1]
Píxeles stego:      [150 151 149 152 153 155 155]
Secreto recuperado: [1 0 1]
¿Éxito?: Sí


Como se evidencia en los resultados de ejecución, el algoritmo modificó de manera exitosa un único píxel (pasando el penúltimo valor de 154 a 155) para satisfacer la matriz de paridad. Posteriormente, la función de extracción ciega recuperó el secreto original en su totalidad, validando la integridad del diseño algorítmico y demostrando la eficiencia del Matrix Embedding de primer orden.

## **1.2. Código (15,11)**

### **1.2.1. Escalamiento a Matrix Embedding con Hamming (15,11)**

Para analizar un escenario de mayor eficiencia de inserción, implementamos el código Hamming (15,11). En esta configuración, la matriz de comprobación de paridad $H$ tiene dimensiones de $4 \times 15$. Esto nos permite ocultar 4 bits de información en un bloque de 15 píxeles, realizando como máximo una sola modificación.

Teóricamente, esto ofrece un mejor compromiso (trade-off) entre la carga útil y la distorsión estadística, ya que la probabilidad de alterar un píxel disminuye proporcionalmente en bloques más grandes, lo cual evaluaremos frente a las redes residuales.

In [7]:
def embed_hamming_15_11(pixels_block, secret_bits):
    """
    Oculta 4 bits secretos en un bloque de 15 píxeles usando Matrix Embedding.
    pixels_block: array de numpy con 15 valores (ej. [150, 151, ... 164])
    secret_bits: array de numpy con 4 bits (ej. [1, 0, 1, 1])
    """
    # 1. Matriz H de 4x15. Las columnas representan los números del 1 al 15 en binario.
    H = np.array([
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], # Fila de los 8s
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1], # Fila de los 4s
        [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1], # Fila de los 2s
        [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]  # Fila de los 1s
    ])

    x = pixels_block % 2
    s_current = np.dot(H, x) % 2
    d = (secret_bits - s_current) % 2

    # 2. Encontrar el índice a modificar (conversión de 4 bits a decimal)
    index_to_change = d[0]*8 + d[1]*4 + d[2]*2 + d[3]*1

    modified_pixels = pixels_block.copy()

    # 3. Modificar el píxel correspondiente si es necesario
    if index_to_change > 0:
        idx = index_to_change - 1

        if modified_pixels[idx] % 2 == 0:
            modified_pixels[idx] += 1
        else:
            modified_pixels[idx] -= 1

    return modified_pixels

### **1.2.2. Extracción del Mensaje con Hamming (15,11)**

De manera análoga al esquema de menor orden, la recuperación de los datos incrustados mediante Hamming (15,11) se realiza de forma ciega, lo que significa que el receptor no requiere acceso a la imagen de cobertura original para extraer el payload.

El proceso de extracción consiste en segmentar la imagen stego en bloques independientes de 15 píxeles, aislar sus bits menos significativos (LSB) y calcular el producto punto de este vector contra la matriz de comprobación de paridad $H$ de dimensiones $4 \times 15$. El vector resultante de 4 bits corresponde al síndrome del bloque, el cual, por la propiedad del Matrix Embedding, es matemáticamente equivalente al fragmento del mensaje secreto original.

In [8]:
def extract_hamming_15_11(stego_pixels_block):
    """
    Extrae los 4 bits secretos ocultos en un bloque de 15 píxeles stego.
    """
    H = np.array([
        [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
        [0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
        [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
    ])

    x_stego = stego_pixels_block % 2
    extracted_bits = np.dot(H, x_stego) % 2

    return extracted_bits

### **Validación Funcional: Inserción y Extracción con Hamming (15,11)**

De forma equivalente, se evaluó el comportamiento del esquema de mayor orden. Al incrustar un payload de 4 bits ([1, 1, 0, 1]) en un bloque continuo de 15 píxeles, el motor de inserción determinó matemáticamente que solo era necesario alterar un píxel (modificando el valor 132 por 133) para que el nuevo síndrome del bloque coincidiera exactamente con el secreto.


In [9]:
# --- Bloque de Prueba Unitaria ---
print("--- Prueba Hamming (15,11) ---")
original_pixels_15 = np.array([120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134])
secret_4bits = np.array([1, 1, 0, 1])

stego_pixels_15 = embed_hamming_15_11(original_pixels_15, secret_4bits)
recovered_secret_15 = extract_hamming_15_11(stego_pixels_15)

print(f"Píxeles originales: {original_pixels_15}")
print(f"Secreto a ocultar:  {secret_4bits}")
print(f"Píxeles stego:      {stego_pixels_15}")
print(f"Secreto recuperado: {recovered_secret_15}")
print(f"¿Éxito?: {'Sí' if np.array_equal(secret_4bits, recovered_secret_15) else 'No'}")

--- Prueba Hamming (15,11) ---
Píxeles originales: [120 121 122 123 124 125 126 127 128 129 130 131 132 133 134]
Secreto a ocultar:  [1 1 0 1]
Píxeles stego:      [120 121 122 123 124 125 126 127 128 129 130 131 133 133 134]
Secreto recuperado: [1 1 0 1]
¿Éxito?: Sí


La recuperación demostró ser 100% precisa, confirmando que la lógica de prevención de desbordamientos de rango para los LSB funciona sin corromper la información subyacente.

## **1.3. Conclusión de la Fase 1: Implementación de Algoritmos**

La primera fase experimental del proyecto confirma que la implementación de los códigos Hamming (7,4) y (15,11) en el dominio espacial es computacionalmente precisa. Ambos algoritmos logran ocultar flujos de bits minimizando la tasa de alteración a un máximo estricto de una modificación por bloque, lo cual garantiza la alta eficiencia de incrustación que fundamenta la técnica de Matrix Embedding.

Al validar que la matemática de los síndromes opera correctamente y que los algoritmos manejan la inversión de los bits menos significativos (LSB) sin inducir errores de rango en la escala de grises, se establece una base algorítmica robusta. Esto permite dar por concluida la implementación criptográfica y avanzar con garantías hacia el procesamiento de imágenes a gran escala, la generación del corpus experimental y la posterior confrontación contra el esteganálisis basado en redes residuales.

# **Fase 2: Procesamiento de Imágenes y Generación de Dataset**

## **2.1. Motor de Inserción y Procesamiento de Imágenes**

Para escalar la técnica de Matrix Embedding a imágenes completas, se desarrolló un motor de procesamiento que opera en el dominio espacial. El flujo de trabajo consiste en:

1. Conversión a Escala de Grises: Se trabaja sobre un solo canal para simplificar la demostración de la eficiencia de los códigos Hamming.

2. Serialización del Payload: El mensaje secreto se convierte en un flujo de bits aleatorios.

3. Segmentación por Bloques: La imagen se aplanada y se divide en bloques de tamaño $n$ (7 o 15), aplicando la modificación de LSB correspondiente según el síndrome calculado.

4. Preservación de Integridad: Los resultados se almacenan en formato PNG (Lossless) para evitar que la compresión por transformada de coseno (como en JPEG) destruya la información oculta en los bits menos significativos.

In [10]:
def process_full_image(image_path, n_type=7):
    """
    Carga una imagen, oculta un mensaje aleatorio y la guarda.
    n_type: 7 para Hamming(7,4) o 15 para Hamming(15,11)
    """
    # 1. Cargar imagen en escala de grises
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Error: No se pudo cargar {image_path}")
        return

    h, w = img.shape
    pixels = img.flatten() # Aplanar para iterar fácilmente

    # Definir parámetros según el tipo de Hamming seleccionado
    if n_type == 7:
        n, k_bits = 7, 3
        embed_func = embed_hamming_7_3
    else:
        n, k_bits = 15, 4
        embed_func = embed_hamming_15_11

    # 2. Calcular capacidad y generar secreto aleatorio
    num_blocks = len(pixels) // n
    total_secret_bits = num_blocks * k_bits
    # Generamos un secreto aleatorio para la prueba (simulando datos cifrados)
    secret_payload = np.random.randint(0, 2, total_secret_bits)

    # 3. Iterar por bloques y aplicar embedding
    stego_pixels = pixels.copy()
    for i in range(num_blocks):
        start_px = i * n
        start_bit = i * k_bits

        block = pixels[start_px : start_px + n]
        bits = secret_payload[start_bit : start_bit + k_bits]

        stego_pixels[start_px : start_px + n] = embed_func(block, bits)

    # 4. Reconstruir la imagen 2D
    stego_img = stego_pixels.reshape((h, w))

    # 5. Guardar resultado en formato PNG (Crucial: NO usar JPG)
    output_name = f"stego_{n_type}_{os.path.basename(image_path)}"
    cv2.imwrite(output_name, stego_img)

    print(f"Imagen generada con éxito: {output_name}")
    print(f"Bits ocultos: {total_secret_bits} | Eficiencia: 1 cambio máx por bloque de {n} px.")
    return output_name

## **2.2. Ejecución y Análisis de Capacidad (Prueba Piloto)**

Para validar el funcionamiento del motor de inserción y observar el comportamiento de las tasas de incrustación espacial sin depender de recursos externos, se genera dinámicamente una imagen de control sintética de $256 \times 256$ píxeles.

Sobre esta matriz controlada se aplican ambos esquemas de Matrix Embedding, haciendo evidente el compromiso inherente entre la longitud del bloque y la capacidad de carga útil (payload):

* Esquema Hamming (7,4): Tasa teórica de $\approx 0.43$ bits por píxel.

* Esquema Hamming (15,11): Tasa teórica de $\approx 0.27$ bits por píxel.

Esta prueba aísla la lógica matemática de los síndromes antes de escalar el procesamiento al corpus completo.

In [11]:
print("--- 1. Generación de Imagen de Control ---")
# Creamos una imagen sintética de 256x256 con ruido aleatorio (simulando texturas)
imagen_sintetica = np.random.randint(0, 256, (256, 256), dtype=np.uint8)
ruta_prueba = 'imagen_de_prueba_sintetica.png'
cv2.imwrite(ruta_prueba, imagen_sintetica)
print(f"Imagen temporal creada: {ruta_prueba} (Dimensiones: 256x256)")

print("\n--- 2. Prueba de Inserción con Hamming (7,4) ---")
img_7_path = process_full_image(ruta_prueba, n_type=7)

print("\n--- 3. Prueba de Inserción con Hamming (15,11) ---")
img_15_path = process_full_image(ruta_prueba, n_type=15)

--- 1. Generación de Imagen de Control ---
Imagen temporal creada: imagen_de_prueba_sintetica.png (Dimensiones: 256x256)

--- 2. Prueba de Inserción con Hamming (7,4) ---
Imagen generada con éxito: stego_7_imagen_de_prueba_sintetica.png
Bits ocultos: 28086 | Eficiencia: 1 cambio máx por bloque de 7 px.

--- 3. Prueba de Inserción con Hamming (15,11) ---
Imagen generada con éxito: stego_15_imagen_de_prueba_sintetica.png
Bits ocultos: 17476 | Eficiencia: 1 cambio máx por bloque de 15 px.


El código (7,4) permite ocultar un volumen significativamente mayor de información que el código (15,11) para las mismas dimensiones espaciales. Sin embargo, esta mayor capacidad está intrínsecamente ligada a una mayor densidad de modificaciones, lo que se evaluará estadísticamente en la siguiente fase.

In [12]:
# Limpieza de Artefactos de Prueba
archivos_a_eliminar = [ruta_prueba, img_7_path, img_15_path]

print("--- Intentando eliminar artefactos de prueba ---")
for archivo in archivos_a_eliminar:
    if os.path.exists(archivo):
        os.remove(archivo)
        print(f"Artefacto temporal eliminado: {archivo}")
    else:
        print(f"Artefacto {archivo} no encontrado (ya eliminado o nunca creado).")


# Verificar que los archivos hayan sido eliminados
print("\n--- Verificación de eliminación ---")
for archivo in archivos_a_eliminar:
    if not os.path.exists(archivo):
        print(f"Verificado: {archivo} ha sido eliminado exitosamente.")
    else:
        print(f"Error: {archivo} aún existe.")

--- Intentando eliminar artefactos de prueba ---
Artefacto temporal eliminado: imagen_de_prueba_sintetica.png
Artefacto temporal eliminado: stego_7_imagen_de_prueba_sintetica.png
Artefacto temporal eliminado: stego_15_imagen_de_prueba_sintetica.png

--- Verificación de eliminación ---
Verificado: imagen_de_prueba_sintetica.png ha sido eliminado exitosamente.
Verificado: stego_7_imagen_de_prueba_sintetica.png ha sido eliminado exitosamente.
Verificado: stego_15_imagen_de_prueba_sintetica.png ha sido eliminado exitosamente.


## **2.3. Adquisición y Preparación del Dataset BOSSBase**

Para garantizar la validez científica del esteganálisis, se optó por utilizar BOSSBase 1.01 (Break Our Steganographic System), un corpus estandarizado en la literatura de seguridad de la información. Este conjunto de datos consta de imágenes en formato PGM (Portable Gray Map) con texturas ricas y ruido de sensor natural, lo que las hace ideales como imágenes de cobertura.

Mediante el siguiente bloque, se automatiza la descarga directa del repositorio académico y la extracción de las muestras en el entorno de ejecución, conformando el directorio raíz de imágenes limpias (cover) que alimentará el motor de inserción de Matrix Embedding.

In [35]:
# --- 1. Descarga del Dataset Oficial (BOSSBase 1.01) ---
# Enlace actualizado con la ruta /ImageDB/
print("1. Descargando BOSSBase 1.01 (Puede tardar unos minutos, son ~1.5 GB)...")
!wget -q http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip -O bossbase.zip

1. Descargando BOSSBase 1.01 (Puede tardar unos minutos, son ~1.5 GB)...


"wget" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [36]:
# --- 2. Extracción de los archivos ---
print("Descomprimiendo el archivo ZIP...")
!unzip -q bossbase.zip -d ./bossbase_completo/

Descomprimiendo el archivo ZIP...


"unzip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [14]:
# --- 3. Traslado a nuestra estructura de directorios ---
# Creamos la carpeta donde tu script buscará las imágenes
directorio_cover = "./dataset/cover/"
os.makedirs(directorio_cover, exist_ok=True)

In [38]:
# BOSSBase viene en formato .pgm
rutas_bossbase = glob.glob("./bossbase_completo/BOSSbase_1.01/*.pgm")

print(f"Se encontraron {len(rutas_bossbase)} imágenes en BOSSBase.")

Se encontraron 10000 imágenes en BOSSBase.


In [39]:
# Dependiendo de la RAM se puede cambiar la cantidad de img a procesar
CANTIDAD_A_PROCESAR = 1000

print(f"3. Copiando {CANTIDAD_A_PROCESAR} imágenes a la carpeta de trabajo.")
for img_path in rutas_bossbase[:CANTIDAD_A_PROCESAR]:
    shutil.copy(img_path, directorio_cover)

3. Copiando 1000 imágenes a la carpeta de trabajo.


In [15]:
# Verificación de cantidad de imágenes en covertura
print(f"¡Dataset listo! Tenemos {len(os.listdir(directorio_cover))} imágenes reales de cobertura preparadas.")

¡Dataset listo! Tenemos 1000 imágenes reales de cobertura preparadas.


## **2.4. Generación del Corpus Esteganográfico (Procesamiento por Lotes)**

Una vez consolidado el directorio base con las imágenes oficiales del corpus BOSSBase, se procede a la generación masiva de los conjuntos de datos modificados.

Se definió una función iterativa que recorre la totalidad de las imágenes de cobertura, extrae sus matrices de píxeles y aplica, de manera independiente, las rutinas de Matrix Embedding para los códigos Hamming (7,4) y (15,11). Las imágenes resultantes se exportan forzosamente en formato PNG (Portable Network Graphics) dentro de sus respectivos directorios (stego_7 y stego_15). Esta conversión de formato es una medida crítica para asegurar una compresión sin pérdidas (lossless), previniendo así la destrucción de los bits menos significativos (LSB) que contienen el payload incrustado.

In [16]:
# --- 1. Definición de la Función Principal ---
def generate_stego_dataset(input_dir, output_dir_7, output_dir_15, max_images=None):
    """
    Recorre una carpeta de imágenes originales y genera dos datasets stego completos.
    """
    os.makedirs(output_dir_7, exist_ok=True)
    os.makedirs(output_dir_15, exist_ok=True)

    image_paths = glob.glob(os.path.join(input_dir, '*.*'))

    if max_images:
        image_paths = image_paths[:max_images]

    print(f"Iniciando procesamiento de {len(image_paths)} imágenes. Esto puede tardar varios minutos...")

    for idx, img_path in enumerate(image_paths):
        filename = os.path.basename(img_path)
        name_without_ext = os.path.splitext(filename)[0]
        output_name = f"{name_without_ext}.png" # Forzamos PNG sin pérdidas

        path_out_7 = os.path.join(output_dir_7, output_name)
        path_out_15 = os.path.join(output_dir_15, output_name)

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue

        h, w = img.shape
        pixels = img.flatten()

        # --- Procesamiento Hamming (7,4) ---
        num_blocks_7 = len(pixels) // 7
        secret_7 = np.random.randint(0, 2, num_blocks_7 * 3)
        stego_pixels_7 = pixels.copy()

        for i in range(num_blocks_7):
            start_px, start_bit = i * 7, i * 3
            block = pixels[start_px : start_px + 7]
            bits = secret_7[start_bit : start_bit + 3]
            stego_pixels_7[start_px : start_px + 7] = embed_hamming_7_3(block, bits)

        cv2.imwrite(path_out_7, stego_pixels_7.reshape((h, w)))

        # --- Procesamiento Hamming (15,11) ---
        num_blocks_15 = len(pixels) // 15
        secret_15 = np.random.randint(0, 2, num_blocks_15 * 4)
        stego_pixels_15 = pixels.copy()

        for i in range(num_blocks_15):
            start_px, start_bit = i * 15, i * 4
            block = pixels[start_px : start_px + 15]
            bits = secret_15[start_bit : start_bit + 4]
            stego_pixels_15[start_px : start_px + 15] = embed_hamming_15_11(block, bits)

        cv2.imwrite(path_out_15, stego_pixels_15.reshape((h, w)))

        # Imprimir progreso cada 100 imágenes para no saturar la consola
        if (idx + 1) % 100 == 0 or (idx + 1) == len(image_paths):
            print(f"Progreso: {idx + 1}/{len(image_paths)} imágenes procesadas y guardadas.")

    print("\n¡Generación del corpus stego finalizada con éxito!")

### **2.4.1. Procesamiento Exhaustivo del Corpus BOSSBase**

Habiendo validado la operatividad del flujo de directorios y la integridad matemática del motor de inserción en la prueba piloto, se procede con el procesamiento masivo del dataset fotográfico oficial (BOSSBase 1.01).

El siguiente bloque de código ejecuta el pipeline iterativo sobre la totalidad de las 10,000 muestras de cobertura. El algoritmo lee secuencialmente las rutas, aplica de manera independiente las incrustaciones de Matrix Embedding para los códigos Hamming (7,4) y (15,11), y exporta las imágenes stego resultantes en los directorios correspondientes. Se fuerza la salida a formato PNG para garantizar una compresión sin pérdidas (lossless) y preservar los LSB. Para auditar la estabilidad computacional del entorno de ejecución, el sistema reporta el progreso en lotes. La culminación de este proceso consolida el corpus definitivo que alimentará la fase de entrenamiento del modelo de esteganálisis residual.

In [42]:
# --- 3. Ejecución sobre el Corpus BOSSBase ---
directorio_originales = "./dataset/cover/"
directorio_stego_7 = "./dataset/stego_7/"
directorio_stego_15 = "./dataset/stego_15/"

In [43]:
# Compilación del procesamiento y guardado de img
generate_stego_dataset(
    input_dir=directorio_originales,
    output_dir_7=directorio_stego_7,
    output_dir_15=directorio_stego_15
)

Iniciando procesamiento de 1000 imágenes. Esto puede tardar varios minutos...
Progreso: 100/1000 imágenes procesadas y guardadas.
Progreso: 200/1000 imágenes procesadas y guardadas.
Progreso: 300/1000 imágenes procesadas y guardadas.
Progreso: 400/1000 imágenes procesadas y guardadas.
Progreso: 500/1000 imágenes procesadas y guardadas.
Progreso: 600/1000 imágenes procesadas y guardadas.
Progreso: 700/1000 imágenes procesadas y guardadas.
Progreso: 800/1000 imágenes procesadas y guardadas.
Progreso: 900/1000 imágenes procesadas y guardadas.
Progreso: 1000/1000 imágenes procesadas y guardadas.

¡Generación del corpus stego finalizada con éxito!


# **Fase 3: Entrenamiento del Modelo de esteganálisis**

## **3.1. Construcción del Pipeline de Datos y Partición**

Para la fase de Deep Learning, se optó por el framework Keras/TensorFlow debido a su eficiencia en la estructuración de pipelines de datos espaciales. Utilizando las utilidades de carga directa desde directorios, se consolidó un flujo de tensores que extrae las imágenes de cobertura y las imágenes stego directamente desde la estructura de archivos generada en la Fase 2.

Para garantizar una evaluación científica rigurosa, se implementó en esta etapa la partición del corpus en dos subconjuntos: 80% para entrenamiento y 20% para validación, fijando una semilla aleatoria para asegurar la reproducibilidad. Adicionalmente, se integraron rutinas de optimización en memoria (cache y prefetch) para mitigar cuellos de botella en las operaciones de lectura de disco.

Consideración Crítica: Se configuró el motor de carga para leer las imágenes en escala de grises y se inhabilitaron las interpolaciones de redimensionamiento, garantizando así que los bits menos significativos (donde reside el payload de los códigos Hamming) permanezcan inalterados antes de ingresar a la red neuronal residual.

In [44]:
print("--- Estandarizando el Corpus Cover a formato PNG ---")

directorio_cover = "./dataset/cover/"
rutas_pgm = glob.glob(os.path.join(directorio_cover, "*.pgm"))

for ruta in rutas_pgm:
    # 1. Leemos la imagen PGM
    img = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
    if img is not None:
        # 2. Creamos la nueva ruta terminada en .png
        nueva_ruta = ruta.replace('.pgm', '.png')
        # 3. Guardamos la imagen en PNG
        cv2.imwrite(nueva_ruta, img)
        # 4. Borramos el archivo PGM viejo para no tener duplicados
        os.remove(ruta)

print(f"¡Saneamiento exitoso! Se convirtieron {len(rutas_pgm)} imágenes a formato PNG.")

--- Estandarizando el Corpus Cover a formato PNG ---


¡Saneamiento exitoso! Se convirtieron 1000 imágenes a formato PNG.


In [17]:
print("--- 1. Definición de Parámetros Dinámicos ---")
directorio_base = "./dataset/"
directorio_cover = os.path.join(directorio_base, "cover")

# Dimensiones dinámicas (BOSSBase utiliza resoluciones nativas de 512x512)
lista_imagenes = glob.glob(os.path.join(directorio_cover, '*.png'))
if lista_imagenes:
    imagen_muestra = cv2.imread(lista_imagenes[0], cv2.IMREAD_GRAYSCALE)
    ALTO_IMG, ANCHO_IMG = imagen_muestra.shape
else:
    ALTO_IMG, ANCHO_IMG = 512, 512

BATCH_SIZE = 32
SEMILLA = 42

print(f"Dimensiones de entrada configuradas a: {ALTO_IMG}x{ANCHO_IMG}")

--- 1. Definición de Parámetros Dinámicos ---
Dimensiones de entrada configuradas a: 512x512


In [18]:
print("\n--- 2. Partición de Datos (80% Entrenamiento / 20% Validación) ---")
train_dataset = image_dataset_from_directory(
    directorio_base,
    validation_split=0.2,
    subset="training",
    seed=SEMILLA,
    labels='inferred',
    label_mode='binary',
    class_names=['cover', 'stego_15'], # Entrenamos contra Hamming(15,11)
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    image_size=(ALTO_IMG, ANCHO_IMG),
    interpolation='nearest' # Vital: sin redimensionamiento destructivo
)

val_dataset = image_dataset_from_directory(
    directorio_base,
    validation_split=0.2,
    subset="validation",
    seed=SEMILLA,
    labels='inferred',
    label_mode='binary',
    class_names=['cover', 'stego_15'],
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    image_size=(ALTO_IMG, ANCHO_IMG),
    interpolation='nearest'
)


--- 2. Partición de Datos (80% Entrenamiento / 20% Validación) ---
Found 2000 files belonging to 2 classes.
Using 1600 files for training.
Found 2000 files belonging to 2 classes.
Using 400 files for validation.


### **3.1.1. Interpretación del Flujo de Tensores**

La validación inicial del Data Loader confirma que Keras ha construido correctamente las estructuras multidimensionales (tensores) requeridas para la ingesta de la red convolucional.

In [19]:
print("\n--- 3. Optimización de Memoria (I/O) ---")

# Inferencia de clases:
for images, labels in train_dataset.take(1):
    print(f"\n[Entrenamiento] Forma del lote de imágenes: {images.shape}")


AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# Extracción de un lote para comprobación (Sección 3.1.1)
for imagenes, etiquetas in train_dataset.take(1):
    print(f"\n[Validación] Forma del lote de imágenes: {imagenes.shape}")
    print(f"[Validación] Etiquetas del lote (primeras 5): {etiquetas.numpy()[:5]}")

# batch size
print(f"\n[Validación] Tamaño del lote: {BATCH_SIZE}")


--- 3. Optimización de Memoria (I/O) ---

[Entrenamiento] Forma del lote de imágenes: (32, 512, 512, 1)

[Validación] Forma del lote de imágenes: (32, 512, 512, 1)
[Validación] Etiquetas del lote (primeras 5): [[0.]
 [0.]
 [1.]
 [0.]
 [0.]]

[Validación] Tamaño del lote: 32


- **Inferencia de Clases (['cover', 'stego_15']):** El motor asigna valores binarios basándose en el orden alfabético de los directorios. La clase cover se mapea a la etiqueta 0.0 y la clase stego_15 a la etiqueta 1.0.

- **Topología del Lote ((32, 512, 512, 1)):** Este tensor de cuarto orden define la estructura de entrada:

- **32 (Batch Size):** Representa el número de muestras procesadas en paralelo por cada paso del gradiente.

- **512, 512 (Resolución Espacial):** Confirma la extracción nativa de las dimensiones de BOSSBase sin interpolaciones destructivas.

- **1 (Canales):** Indica la profundidad de intensidad en escala de grises.-

- **Mapeo de Etiquetas:** Se verifica que el vector de salida del lote contenga una distribución estocástica de ceros y unos, confirmando que el proceso de shuffle está operativo y que la red recibirá muestras balanceadas en cada iteración.

## **3.2. Diseño de la Arquitectura Residual**

A diferencia de las tareas tradicionales de visión por computadora, donde el objetivo es extraer características semánticas (bordes, formas, objetos), el esteganálisis requiere identificar alteraciones de altísima frecuencia correspondientes al ruido inducido en los bits menos significativos (LSB).

Para este fin, se diseñó una arquitectura de red neuronal convolucional (CNN) adaptada mediante el uso de bloques residuales. Las conexiones de salto (skip connections) introducidas en estos bloques previenen la degradación del gradiente durante el entrenamiento y permiten que la red retenga las señales débiles del ruido esteganográfico a lo largo de las capas de extracción de características.

La red culmina en una capa de agrupamiento global (Global Average Pooling) seguida de un perceptrón con función de activación sigmoide, optimizada para entregar una predicción binaria continua entre 0 (Cobertura) y 1 (Stego).

In [20]:
# ============================================================
# PREPROCESAMIENTO: Filtros Pasa-Altos (HPF)
# Versión corregida para evitar colapso a una sola clase
# ============================================================

_k1 = np.array([[-1,  2, -1],
                [ 2, -4,  2],
                [-1,  2, -1]], dtype=np.float32) / 4.0

_k2 = np.array([[ 1,  0, -1],
                [ 2,  0, -2],
                [ 1,  0, -1]], dtype=np.float32) / 4.0

_k3 = np.array([[ 1,  2,  1],
                [ 0,  0,  0],
                [-1, -2, -1]], dtype=np.float32) / 4.0

HPF_WEIGHTS = np.stack([_k1, _k2, _k3], axis=-1)[:, :, np.newaxis, :]


# ============================================================
# BLOQUE RESIDUAL CORREGIDO
# Cambios importantes:
# - LeakyReLU en lugar de ReLU para no matar residuos negativos
# - SE desactivado por defecto al inicio
# - menos regularización agresiva
# ============================================================

def bloque_residual_estego(x, filtros, usar_se=False, nombre="bloque"):
    salto = x

    y = layers.Conv2D(
        filtros,
        (3, 3),
        padding="same",
        use_bias=False,
        name=f"{nombre}_conv1"
    )(x)
    y = layers.BatchNormalization(name=f"{nombre}_bn1")(y)
    y = layers.LeakyReLU(negative_slope=0.1, name=f"{nombre}_lrelu1")(y)

    y = layers.Conv2D(
        filtros,
        (3, 3),
        padding="same",
        use_bias=False,
        name=f"{nombre}_conv2"
    )(y)
    y = layers.BatchNormalization(name=f"{nombre}_bn2")(y)

    if usar_se:
        num_filtros = y.shape[-1]
        se = layers.GlobalAveragePooling2D(name=f"{nombre}_se_gap")(y)
        se = layers.Reshape((1, 1, num_filtros), name=f"{nombre}_se_reshape")(se)
        se = layers.Dense(
            max(num_filtros // 8, 4),
            activation="relu",
            use_bias=False,
            name=f"{nombre}_se_dense1"
        )(se)
        se = layers.Dense(
            num_filtros,
            activation="sigmoid",
            use_bias=False,
            name=f"{nombre}_se_dense2"
        )(se)
        y = layers.Multiply(name=f"{nombre}_se_mult")([y, se])

    if salto.shape[-1] != filtros:
        salto = layers.Conv2D(
            filtros,
            (1, 1),
            padding="same",
            use_bias=False,
            name=f"{nombre}_proj"
        )(salto)
        salto = layers.BatchNormalization(name=f"{nombre}_proj_bn")(salto)

    y = layers.Add(name=f"{nombre}_add")([salto, y])
    y = layers.LeakyReLU(negative_slope=0.1, name=f"{nombre}_out")(y)

    return y


# ============================================================
# FUNCIÓN PARA CONSTRUIR EL MODELO
# Sirve tanto para Hamming (15,11) como para Hamming (7,4)
# ============================================================

def construir_modelo_estego(nombre_modelo="ResNetEstegoCorregida"):
    entrada = Input(shape=(ALTO_IMG, ANCHO_IMG, 1), name="imagen_entrada")

    # OJO:
    # No se divide entre 255 antes del HPF.
    # La señal LSB ya es muy pequeña; escalar antes del HPF la debilita demasiado.
    x = layers.Conv2D(
        3,
        (3, 3),
        padding="same",
        kernel_initializer=tf.keras.initializers.Constant(HPF_WEIGHTS),
        use_bias=False,
        trainable=False,
        name="filtro_hpf_fijo"
    )(entrada)

    # Truncamiento tipo SRM: evita valores extremos y estabiliza el entrenamiento.
    x = layers.Lambda(
        lambda t: tf.clip_by_value(t, -3.0, 3.0),
        name="truncamiento_hpf"
    )(x)

    x = layers.BatchNormalization(name="bn_hpf")(x)

    # Extracción inicial
    x = layers.Conv2D(
        32,
        (3, 3),
        padding="same",
        use_bias=False,
        name="conv_inicial"
    )(x)
    x = layers.BatchNormalization(name="bn_inicial")(x)
    x = layers.LeakyReLU(negative_slope=0.1, name="lrelu_inicial")(x)

    # Bloques residuales más moderados
    x = bloque_residual_estego(x, 32, usar_se=False, nombre="res1")
    x = layers.AveragePooling2D((2, 2), name="avgpool1")(x)

    x = bloque_residual_estego(x, 64, usar_se=False, nombre="res2")
    x = layers.AveragePooling2D((2, 2), name="avgpool2")(x)

    x = bloque_residual_estego(x, 128, usar_se=True, nombre="res3")
    x = layers.AveragePooling2D((2, 2), name="avgpool3")(x)

    x = bloque_residual_estego(x, 128, usar_se=True, nombre="res4")

    # Cabeza clasificadora menos agresiva
    # Quitamos GMP por ahora porque puede amplificar outliers y favorecer colapso.
    x = layers.GlobalAveragePooling2D(name="gap")(x)

    x = layers.Dense(
        64,
        activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-5),
        name="dense_64"
    )(x)
    x = layers.Dropout(0.2, name="dropout_1")(x)

    salida = layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="prediccion_stego"
    )(x)

    modelo = models.Model(
        inputs=entrada,
        outputs=salida,
        name=nombre_modelo
    )

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return modelo


print("--- Ensamblando Arquitectura Corregida para Hamming (15,11) ---")

model_resnet = construir_modelo_estego(
    nombre_modelo="ResNetEstego_Hamming15_Corregida"
)

model_resnet.summary()

--- Ensamblando Arquitectura Corregida para Hamming (15,11) ---



Model: "ResNetEstego_Hamming15_Corregida"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ imagen_entrada      │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ filtro_hpf_fijo     │ (None, 512, 512,  │         27 │ imagen_entrada[0… │
│ (Conv2D)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ truncamiento_hpf    │ (None, 512, 512,  │          0 │ filtro_hpf_fijo[… │
│ (Lambda)            │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_hpf              │ (None, 512, 512,  │         12 │ truncamiento_hpf… │
│ (BatchNormalizatio… │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_inicial        │ (None, 512, 512,  │        864 │ bn_hpf[0][0]      │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_inicial          │ (None, 512, 512,  │        128 │ conv_inicial[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lrelu_inicial       │ (None, 512, 512,  │          0 │ bn_inicial[0][0]  │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_conv1 (Conv2D) │ (None, 512, 512,  │      9,216 │ lrelu_inicial[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_bn1            │ (None, 512, 512,  │        128 │ res1_conv1[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_lrelu1         │ (None, 512, 512,  │          0 │ res1_bn1[0][0]    │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_conv2 (Conv2D) │ (None, 512, 512,  │      9,216 │ res1_lrelu1[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_bn2            │ (None, 512, 512,  │        128 │ res1_conv2[0][0]  │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_add (Add)      │ (None, 512, 512,  │          0 │ lrelu_inicial[0]… │
│                     │ 32)               │            │ res1_bn2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res1_out            │ (None, 512, 512,  │          0 │ res1_add[0][0]    │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ avgpool1            │ (None, 256, 256,  │          0 │ res1_out[0][0]    │
│ (AveragePooling2D)  │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res2_conv1 (Conv2D) │ (None, 256, 256,  │     18,432 │ avgpool1[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ res2_bn1            │ (None, 256, 256,  │        256 │ res2_conv1[0][0]

 Total params: 621,192 (2.37 MB)

 Trainable params: 619,303 (2.36 MB)

 Non-trainable params: 1,889 (7.38 KB)

### **3.3. Configuración y Protocolo de Entrenamiento**

Una vez consolidada la arquitectura y el flujo de datos, se establece el protocolo de optimización. El entrenamiento se rige por una función de pérdida de entropía cruzada binaria (Binary Crossentropy), adecuada para la naturaleza del problema.  

Para garantizar la convergencia y mitigar el riesgo de sobreajuste (overfitting) en un escenario de señales débiles como el esteganálisis, se integran mecanismos de regularización dinámica (callbacks):  

* Early Stopping: Monitorea la pérdida en el conjunto de validación y detiene el proceso si no hay una mejora significativa tras un número determinado de épocas.

* ReduceLROnPlateau: Reduce el factor de aprendizaje (Learning Rate) cuando la pérdida se estanca, permitiendo una búsqueda más fina del mínimo global en el espacio de parámetros.

* Model Checkpoint: Almacena automáticamente los pesos de la época que reporte el mejor desempeño en validación.

In [21]:
# ============================================================
# CALLBACKS PARA HAMMING (15,11)
# ============================================================

callback_early_stop = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

callback_reduce_lr = ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

callback_checkpoint = ModelCheckpoint(
    filepath="mejor_modelo_stego_15.keras",
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)


# ============================================================
# CALLBACK DE DIAGNÓSTICO
# Sirve para detectar si el modelo está prediciendo todo 0 o todo 1
# ============================================================

class PredStatsCallback(tf.keras.callbacks.Callback):
    def __init__(self, dataset, nombre="validacion", n_batches=4):
        super().__init__()
        self.dataset = dataset
        self.nombre = nombre
        self.n_batches = n_batches

    def on_epoch_end(self, epoch, logs=None):
        probs = []

        for i, (x_batch, _) in enumerate(self.dataset.take(self.n_batches)):
            p = self.model.predict(x_batch, verbose=0).reshape(-1)
            probs.extend(p)

        probs = np.array(probs)

        print(
            f"\n[{self.nombre}] "
            f"pred_mean={probs.mean():.4f} | "
            f"pred_min={probs.min():.4f} | "
            f"pred_max={probs.max():.4f} | "
            f"% pred > 0.5 = {(probs > 0.5).mean() * 100:.2f}%"
        )


callback_pred_stats_15 = PredStatsCallback(
    val_dataset,
    nombre="Hamming 15,11"
)

In [ ]:
# Ejecución del Entrenamiento
EPOCAS = 30

historial = model_resnet.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCAS,
    callbacks=[
        callback_early_stop,
        callback_reduce_lr,
        callback_checkpoint,
        callback_pred_stats_15
    ]
)

Epoch 1/30
 2/50 ━━━━━━━━━━━━━━━━━━━━ 4:33:51 342s/step - accuracy: 0.5234 - auc: 0.4550 - loss: 0.7132

## **3.4. Entrenamiento Comparativo: Detección de Hamming (7,4)**
Para establecer una línea base de comparación y validar la hipótesis de que el esquema (15,11) ofrece mayor resistencia al esteganálisis, se procedió a entrenar una segunda instancia de la arquitectura ResNet.

Esta red paralela fue alimentada exclusivamente con el corpus de imágenes de cobertura y las imágenes modificadas mediante el algoritmo Hamming (7,4). Se mantuvieron estáticos los hiperparámetros, la optimización en memoria y la partición estocástica (80/20) para garantizar que cualquier variación en el rendimiento predictivo (Accuracy, AUC) sea directamente atribuible a la diferencia en las tasas de incrustación de los códigos criptográficos.

In [ ]:
print("--- 1. Preparando Pipeline para Hamming (7,4) ---")

# Reutilizamos las dimensiones y la semilla de la sección 3.1
train_dataset_7 = image_dataset_from_directory(
    directorio_base,
    validation_split=0.2,
    subset="training",
    seed=SEMILLA,
    labels='inferred',
    label_mode='binary',
    class_names=  ['cover', 'stego_7'], #Ahora evaluamos el 7,4
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    image_size=(ALTO_IMG, ANCHO_IMG),
    interpolation='nearest'
)

val_dataset_7 = image_dataset_from_directory(
    directorio_base,
    validation_split=0.2,
    subset="validation",
    seed=SEMILLA,
    labels='inferred',
    label_mode='binary',
    class_names=  ['cover', 'stego_7'],
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    image_size=(ALTO_IMG, ANCHO_IMG),
    interpolation='nearest'
)

# Optimización en memoria
train_dataset_7 = train_dataset_7.cache().shuffle(
    1000,
    seed=SEMILLA,
    reshuffle_each_iteration=True
).prefetch(buffer_size=AUTOTUNE)

val_dataset_7 = val_dataset_7.cache().prefetch(buffer_size=AUTOTUNE)


print("\n--- 2. Construyendo y Compilando Red Paralela (Hamming 7,4) ---")

modelo_resnet_7 = construir_modelo_estego(
    nombre_modelo="ResNetEstego_Hamming7_Corregida"
)

modelo_resnet_7.summary()

In [ ]:
# ============================================================
# CALLBACKS PARA HAMMING (7,4)
# ============================================================

callback_checkpoint_7 = ModelCheckpoint(
    filepath="mejor_modelo_stego_7.keras",
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)

callback_early_stop_7 = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=6,
    restore_best_weights=True,
    verbose=1
)

callback_reduce_lr_7 = ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

callback_pred_stats_7 = PredStatsCallback(
    val_dataset_7,
    nombre="Hamming 7,4"
)


print("\n--- 3. Iniciando Entrenamiento (Hamming 7,4) ---")

historial_7 = modelo_resnet_7.fit(
    train_dataset_7,
    validation_data=val_dataset_7,
    epochs=EPOCAS,
    callbacks=[
        callback_early_stop_7,
        callback_reduce_lr_7,
        callback_checkpoint_7,
        callback_pred_stats_7
    ]
)

In [ ]:
# ============================================================
# DIAGNÓSTICO DE SOBREAJUSTE Y CURVAS DE ENTRENAMIENTO
# ============================================================

def graficar_curvas_entrenamiento(historial, nombre_modelo):
    hist = historial.history
    epocas = range(1, len(hist["loss"]) + 1)

    # ----------------------------
    # Gráfica de pérdida
    # ----------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(epocas, hist["loss"], marker="o", label="Entrenamiento")
    plt.plot(epocas, hist["val_loss"], marker="o", label="Validación")

    mejor_epoca_loss = np.argmin(hist["val_loss"]) + 1
    mejor_val_loss = np.min(hist["val_loss"])

    plt.axvline(
        mejor_epoca_loss,
        linestyle="--",
        label=f"Mejor val_loss: época {mejor_epoca_loss}"
    )

    plt.title(f"Pérdida durante el entrenamiento - {nombre_modelo}")
    plt.xlabel("Época")
    plt.ylabel("Binary Crossentropy Loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    # ----------------------------
    # Gráfica de accuracy
    # ----------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(epocas, hist["accuracy"], marker="o", label="Entrenamiento")
    plt.plot(epocas, hist["val_accuracy"], marker="o", label="Validación")

    mejor_epoca_acc = np.argmax(hist["val_accuracy"]) + 1
    mejor_val_acc = np.max(hist["val_accuracy"])

    plt.axvline(
        mejor_epoca_acc,
        linestyle="--",
        label=f"Mejor val_accuracy: época {mejor_epoca_acc}"
    )

    plt.title(f"Accuracy durante el entrenamiento - {nombre_modelo}")
    plt.xlabel("Época")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    # ----------------------------
    # Gráfica de AUC, si existe
    # ----------------------------
    if "auc" in hist and "val_auc" in hist:
        plt.figure(figsize=(8, 5))
        plt.plot(epocas, hist["auc"], marker="o", label="Entrenamiento")
        plt.plot(epocas, hist["val_auc"], marker="o", label="Validación")

        mejor_epoca_auc = np.argmax(hist["val_auc"]) + 1
        mejor_val_auc = np.max(hist["val_auc"])

        plt.axvline(
            mejor_epoca_auc,
            linestyle="--",
            label=f"Mejor val_auc: época {mejor_epoca_auc}"
        )

        plt.title(f"AUC durante el entrenamiento - {nombre_modelo}")
        plt.xlabel("Época")
        plt.ylabel("AUC")
        plt.legend()
        plt.grid(True)
        plt.show()


def diagnosticar_sobreajuste(historial, nombre_modelo):
    hist = historial.history

    train_loss = np.array(hist["loss"])
    val_loss = np.array(hist["val_loss"])

    train_acc = np.array(hist["accuracy"])
    val_acc = np.array(hist["val_accuracy"])

    mejor_epoca_val_loss = np.argmin(val_loss)
    mejor_val_loss = val_loss[mejor_epoca_val_loss]

    ultima_train_loss = train_loss[-1]
    ultima_val_loss = val_loss[-1]

    ultima_train_acc = train_acc[-1]
    ultima_val_acc = val_acc[-1]

    gap_acc = ultima_train_acc - ultima_val_acc

    aumento_val_loss = (ultima_val_loss - mejor_val_loss) / abs(mejor_val_loss)

    print("\n" + "="*70)
    print(f"DIAGNÓSTICO DE SOBREAJUSTE: {nombre_modelo}")
    print("="*70)

    print(f"Mejor época según val_loss: {mejor_epoca_val_loss + 1}")
    print(f"Mejor val_loss: {mejor_val_loss:.4f}")
    print(f"Última train_loss: {ultima_train_loss:.4f}")
    print(f"Última val_loss: {ultima_val_loss:.4f}")
    print(f"Aumento relativo de val_loss respecto al mejor punto: {aumento_val_loss*100:.2f}%")

    print("\nAccuracy final:")
    print(f"Train accuracy: {ultima_train_acc:.4f}")
    print(f"Val accuracy:   {ultima_val_acc:.4f}")
    print(f"Brecha train - val accuracy: {gap_acc:.4f}")

    if "auc" in hist and "val_auc" in hist:
        train_auc = np.array(hist["auc"])
        val_auc = np.array(hist["val_auc"])

        ultima_train_auc = train_auc[-1]
        ultima_val_auc = val_auc[-1]
        gap_auc = ultima_train_auc - ultima_val_auc

        print("\nAUC final:")
        print(f"Train AUC: {ultima_train_auc:.4f}")
        print(f"Val AUC:   {ultima_val_auc:.4f}")
        print(f"Brecha train - val AUC: {gap_auc:.4f}")
    else:
        gap_auc = 0

    print("\nInterpretación:")

    if aumento_val_loss > 0.10 and (gap_acc > 0.08 or gap_auc > 0.08):
        print("Hay señales claras de SOBREAJUSTE.")
        print("El modelo mejora en entrenamiento, pero pierde generalización en validación.")

    elif gap_acc > 0.10 or gap_auc > 0.10:
        print("Hay posible SOBREAJUSTE.")
        print("La diferencia entre entrenamiento y validación es alta.")

    elif ultima_train_acc < 0.60 and ultima_val_acc < 0.60:
        print("No parece sobreajuste fuerte.")
        print("El problema parece más cercano a underfitting, colapso de clase o señal esteganográfica muy débil.")

    else:
        print("No hay señales fuertes de sobreajuste.")
        print("El entrenamiento y la validación se comportan de forma relativamente similar.")


# ============================================================
# EJECUTAR DIAGNÓSTICO PARA AMBOS MODELOS
# ============================================================

graficar_curvas_entrenamiento(
    historial,
    "Hamming (15,11)"
)

diagnosticar_sobreajuste(
    historial,
    "Hamming (15,11)"
)

graficar_curvas_entrenamiento(
    historial_7,
    "Hamming (7,4)"
)

diagnosticar_sobreajuste(
    historial_7,
    "Hamming (7,4)"
)

In [ ]:
# ============================================================
# EVALUACIÓN FINAL: TRAIN VS VALIDATION
# ============================================================

def evaluar_train_vs_val(modelo, train_ds, val_ds, nombre_modelo):
    print("\n" + "="*70)
    print(f"EVALUACIÓN FINAL TRAIN VS VALIDATION: {nombre_modelo}")
    print("="*70)

    resultados_train = modelo.evaluate(
        train_ds,
        verbose=0,
        return_dict=True
    )

    resultados_val = modelo.evaluate(
        val_ds,
        verbose=0,
        return_dict=True
    )

    print("\nResultados en entrenamiento:")
    for metrica, valor in resultados_train.items():
        print(f"{metrica}: {valor:.4f}")

    print("\nResultados en validación:")
    for metrica, valor in resultados_val.items():
        print(f"{metrica}: {valor:.4f}")

    print("\nBrechas train - validation:")
    for metrica in resultados_train.keys():
        brecha = resultados_train[metrica] - resultados_val[metrica]
        print(f"{metrica}: {brecha:.4f}")

    if "accuracy" in resultados_train and "accuracy" in resultados_val:
        gap_accuracy = resultados_train["accuracy"] - resultados_val["accuracy"]

        if gap_accuracy > 0.10:
            print("\nALERTA: posible sobreajuste por brecha alta en accuracy.")

    if "auc" in resultados_train and "auc" in resultados_val:
        gap_auc = resultados_train["auc"] - resultados_val["auc"]

        if gap_auc > 0.10:
            print("\nALERTA: posible sobreajuste por brecha alta en AUC.")


evaluar_train_vs_val(
    model_resnet,
    train_dataset,
    val_dataset,
    "Hamming (15,11)"
)

evaluar_train_vs_val(
    modelo_resnet_7,
    train_dataset_7,
    val_dataset_7,
    "Hamming (7,4)"
)

# **Fase 4: Evaluación y Discusión**

## **4.1. Evaluación de Fidelidad Visual (Métricas de Calidad)**

Para medir el impacto del *Matrix Embedding* en las imágenes de cobertura, se emplearon dos métricas estándar en el procesamiento de imágenes esteganográficas:

1. PSNR (Peak Signal-to-Noise Ratio): Mide la relación entre el máximo valor posible de un píxel y el ruido introducido por el ocultamiento (Error Cuadrático Medio o MSE). Se expresa en decibelios (dB). Un valor más alto indica una menor degradación visual. Su formulación matemática es:

$$MSE = \frac{1}{m\,n} \sum_{i=0}^{m-1} \sum_{j=0}^{n-1} [I(i,j) - K(i,j)]^2$$$$PSNR = 10 \cdot \log_{10} \left( \frac{MAX_I^2}{MSE} \right)$$

2. SSIM (Structural Similarity Index Measure): A diferencia del PSNR, que mide el error absoluto, el SSIM evalúa la degradación de la estructura espacial de la imagen, modelando mejor la percepción del ojo humano. Sus valores oscilan entre $-1$ y $1$, donde $1$ indica una imagen idéntica a la original.

Se espera que el esquema basado en Hamming (15,11) reporte valores de PSNR y SSIM superiores a los de Hamming (7,4) bajo tasas de payload equivalentes, debido a su menor tasa de modificación de píxeles esperada.



In [ ]:
# Rutas de los directorios
dir_cover = "./dataset/cover/"
dir_stego_7 = "./dataset/stego_7/"
dir_stego_15 = "./dataset/stego_15/"

def evaluar_dataset(ruta_cover, ruta_stego, limite=100):
    """
    Compara un conjunto de imágenes limpias vs estego y retorna el promedio de PSNR y SSIM.
    Se utiliza un límite para evaluar una muestra representativa y agilizar el cómputo.
    """
    imagenes_cover = sorted(glob.glob(os.path.join(ruta_cover, '*.png')))[:limite]

    lista_psnr = []
    lista_ssim = []

    for path_cov in imagenes_cover:
        nombre_archivo = os.path.basename(path_cov)
        path_stg = os.path.join(ruta_stego, nombre_archivo)

        if not os.path.exists(path_stg):
            continue

        # Cargar imágenes
        img_cov = cv2.imread(path_cov, cv2.IMREAD_GRAYSCALE)
        img_stg = cv2.imread(path_stg, cv2.IMREAD_GRAYSCALE)

        # Calcular métricas
        val_psnr = compute_psnr(img_cov, img_stg, data_range=255)
        # Se requiere win_size impar y menor que las dimensiones de la imagen
        val_ssim = compute_ssim(img_cov, img_stg, data_range=255, win_size=11)

        lista_psnr.append(val_psnr)
        lista_ssim.append(val_ssim)

    return np.mean(lista_psnr), np.mean(lista_ssim)

In [ ]:
# Evaluamos una muestra estocástica (ej. 100 imágenes) para obtener significancia estadística
MUESTRAS_EVALUACION = 100

psnr_7, ssim_7 = evaluar_dataset(dir_cover, dir_stego_7, limite=MUESTRAS_EVALUACION)
psnr_15, ssim_15 = evaluar_dataset(dir_cover, dir_stego_15, limite=MUESTRAS_EVALUACION)

print(f"\nResultados Promedio sobre {MUESTRAS_EVALUACION} imágenes:")
print("-" * 50)
print(f"Esquema Hamming (7,4):")
print(f" -> PSNR Promedio: {psnr_7:.2f} dB")
print(f" -> SSIM Promedio: {ssim_7:.4f}")
print("-" * 50)
print(f"Esquema Hamming (15,11):")
print(f" -> PSNR Promedio: {psnr_15:.2f} dB")
print(f" -> SSIM Promedio: {ssim_15:.4f}")
print("-" * 50)

# Verificación de hipótesis de tu documento
if psnr_15 > psnr_7:
    print("\n[ÉXITO] La hipótesis se confirma: Hamming (15,11) presenta menor distorsión visual (Mayor PSNR).")

## **4.2. Evaluación de la Red Residual (Métricas de Clasificación)**

Para cuantificar la capacidad de detección del modelo estegoanalítico (ResNet) sobre datos no vistos previamente, se extraen las predicciones probabilísticas utilizando el subconjunto de validación. El desempeño se evalúa mediante dos herramientas estadísticas fundamentales:

1. Matriz de Confusión: Permite visualizar la distribución de los aciertos y errores topológicos del modelo. En el contexto del esteganálisis, resulta crítico analizar la tasa de Falsos Negativos (imágenes con esteganografía que el modelo clasifica erróneamente como limpias), ya que esta métrica cuantifica directamente el nivel de "evasión" exitosa logrado por el algoritmo de Matrix Embedding.

2. Curva ROC (Receiver Operating Characteristic) y AUC (Area Under the Curve): La curva ROC grafica la relación entre la Tasa de Verdaderos Positivos (TPR) y la Tasa de Falsos Positivos (FPR) a través de múltiples umbrales de decisión. El valor del área bajo la curva ($AUC \in [0, 1]$) resume el rendimiento global del clasificador. Un $AUC \approx 0.5$ indica una detección aleatoria (lo que implicaría un esteganálisis fallido y un ocultamiento perfecto), mientras que un $AUC$ cercano a $1.0$ representa una clasificación perfecta.

In [ ]:
print("--- Extrayendo predicciones comparativas de los modelos ---")

def obtener_predicciones(modelo, dataset_val):
    """Extrae las etiquetas reales y las predicciones de probabilidad de un dataset."""
    y_real = []
    y_pred_prob = []
    for img, label in dataset_val:
        y_real.extend(label.numpy())
        preds = modelo.predict(img, verbose=0)
        y_pred_prob.extend(preds)
    return np.array(y_real), np.array(y_pred_prob).flatten()

In [ ]:
# ============================================================
# DIAGNÓSTICO DE COLAPSO DE CLASE
# ============================================================

def diagnosticar_predicciones(nombre, y_real, y_prob):
    y_real = y_real.reshape(-1).astype(int)
    y_prob = y_prob.reshape(-1)

    y_bin = (y_prob > 0.5).astype(int)

    print("\n" + "="*60)
    print(f"DIAGNÓSTICO: {nombre}")
    print("="*60)

    print(f"Clases reales:")
    print(f"  Cover  (0): {np.sum(y_real == 0)}")
    print(f"  Stego  (1): {np.sum(y_real == 1)}")

    print(f"\nPredicciones binarias con umbral 0.5:")
    print(f"  Predicho Cover (0): {np.sum(y_bin == 0)}")
    print(f"  Predicho Stego (1): {np.sum(y_bin == 1)}")

    print(f"\nProbabilidades:")
    print(f"  Media: {y_prob.mean():.4f}")
    print(f"  Mínima: {y_prob.min():.4f}")
    print(f"  Máxima: {y_prob.max():.4f}")
    print(f"  Desviación estándar: {y_prob.std():.4f}")

    if np.all(y_bin == 0):
        print("\nALERTA: el modelo está prediciendo TODO como cover.")
    elif np.all(y_bin == 1):
        print("\nALERTA: el modelo está prediciendo TODO como stego.")
    else:
        print("\nOK: el modelo está usando ambas clases.")

In [ ]:
y_real_15, y_prob_15 = obtener_predicciones(model_resnet, val_dataset)
y_real_7, y_prob_7 = obtener_predicciones(modelo_resnet_7, val_dataset_7)

diagnosticar_predicciones("Hamming (15,11)", y_real_15, y_prob_15)
diagnosticar_predicciones("Hamming (7,4)", y_real_7, y_prob_7)

In [ ]:
# 1. Obtener datos para Hamming (15,11) - Usando tu variable corregida
y_real_15, y_prob_15 = obtener_predicciones(model_resnet, val_dataset)

# 2. Obtener datos para Hamming (7,4)
y_real_7, y_prob_7 = obtener_predicciones(modelo_resnet_7, val_dataset_7)

# 3. Binarización mediante Umbral estándar (0.5)
y_bin_15 = (y_prob_15 > 0.5).astype(int)
y_bin_7 = (y_prob_7 > 0.5).astype(int)

cm_15 = confusion_matrix(y_real_15, y_bin_15)
cm_7 = confusion_matrix(y_real_7, y_bin_7)
# 4. Cálculo de coordenadas para las Curvas ROC y AUC
fpr_15, tpr_15, _ = roc_curve(y_real_15, y_prob_15)
auc_15 = auc(fpr_15, tpr_15)

fpr_7, tpr_7, _ = roc_curve(y_real_7, y_prob_7)
auc_7 = auc(fpr_7, tpr_7)

In [ ]:
# --- 5. DIAGRAMACIÓN VISUAL UNIFICADA ---
plt.figure(figsize=(18, 5))

# Subplot 1: Matriz de Confusión 7,4 (Tonos Rojos para diferenciar)
plt.subplot(1, 3, 1)
sns.heatmap(cm_7, annot=True, fmt='d', cmap='Reds', cbar=False,
            xticklabels=['Cover (0)', 'Stego (1)'],
            yticklabels=['Cover (0)', 'Stego (1)'])
plt.title('Matriz de Confusión: Hamming (7,4)')
plt.ylabel('Etiqueta Real')
plt.xlabel('Predicción del Modelo')

# Subplot 2: Matriz de Confusión 15,11 (Tonos Azules)
plt.subplot(1, 3, 2)
sns.heatmap(cm_15, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Cover (0)', 'Stego (1)'],
            yticklabels=['Cover (0)', 'Stego (1)'])
plt.title('Matriz de Confusión: Hamming (15,11)')
plt.xlabel('Predicción del Modelo')

# Subplot 3: Curvas ROC Superpuestas
plt.subplot(1, 3, 3)
plt.plot(fpr_7, tpr_7, color='darkred', lw=2, label=f'Hamming (7,4) AUC={auc_7:.3f}')
plt.plot(fpr_15, tpr_15, color='darkblue', lw=2, label=f'Hamming (15,11) AUC={auc_15:.3f}')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Detección por Azar')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Análisis ROC: Detección de Señales Esteganográficas')
plt.legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# --- 6. Análisis Estadístico de Evasión ---
fn_7 = cm_7[1, 0]
total_7 = np.sum(cm_7[1, :])
tasa_evasion_7 = (fn_7 / total_7) * 100 if total_7 > 0 else 0

fn_15 = cm_15[1, 0]
total_15 = np.sum(cm_15[1, :])
tasa_evasion_15 = (fn_15 / total_15) * 100 if total_15 > 0 else 0

print("\n" + "="*50)
print("RESUMEN ESTADÍSTICO DE EVASIÓN ESTEGANOGRÁFICA")
print("="*50)
print(f"Esquema (7,4)   - Falsos Negativos: {fn_7}/{total_7} -> Tasa de Evasión: {tasa_evasion_7:.2f}%")
print(f"Esquema (15,11) - Falsos Negativos: {fn_15}/{total_15} -> Tasa de Evasión: {tasa_evasion_15:.2f}%")
print("-" * 50)

if tasa_evasion_15 > tasa_evasion_7:
    print("CONCLUSIÓN: Hipótesis confirmada. El esquema (15,11) es criptográficamente")
    print("más robusto y presenta mayor dificultad de detección forense.")
else:
    print("CONCLUSIÓN: El factor de regularización de la red mitigó el ruido en ambos")
    print("esquemas, logrando tasas de detección similares.")